ML-09 — Validation and Research Claim Audit
Open In Colab

This skeleton is yours to fill. Work the sections in order — each one has a one-line hint. Simple words, honest numbers.

Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

Finding 1 — Growth prediction

The paper reports strong accuracy for predicting which pages would grow or decline. My methodology question is: how was the growing/declining label defined, and was the label measured strictly after the feature window? I would want to confirm that the model could not use information from the outcome period when making its prediction.

Finding 2 — Refresh lift

The paper reports higher impressions for refreshed pages than for stale pages. My methodology question is: how were refreshed and stale pages defined, and were the two groups comparable before the refresh? I would want to understand how selection differences were handled before interpreting the observed difference as evidence of an effect from refreshing.


1. Two paper findings + my methodology questions
Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.

In [11]:
!git clone https://github.com/muska123-web/FlyRank-AI-Internship.git

fatal: destination path 'FlyRank-AI-Internship' already exists and is not an empty directory.


In [12]:
import os

csv_path = "/content/FlyRank-AI-Internship/data/raw/content_refresh_anonymized.csv"

print("CSV exists:", os.path.exists(csv_path))

CSV exists: True


2. My model under an honest split (before/after)
Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.

In [13]:
import pandas as pd

df = pd.read_csv(
    "/content/FlyRank-AI-Internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (30000, 44)


In [14]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Label created successfully!")
print(df["is_declining_label"].value_counts())

Label created successfully!
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [15]:
y = df["is_declining_label"]

X = df.drop(
    columns=[
        "is_declining_label",
        "trend_direction",
        "trend_pct",
        "content_id",
        "client_id"
    ]
)

client_ids = df["client_id"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of features:", X.shape[1])
print("Number of clients:", client_ids.nunique())

X shape: (30000, 40)
y shape: (30000,)
Number of features: 40
Number of clients: 32


In [16]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:", len(categorical_features))
print(categorical_features)

print("\nNumerical features:", len(numerical_features))
print(numerical_features)

Categorical features: 11
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Numerical features: 29
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


In [18]:
# ============================================
# WEEK 6 — BEFORE: Random Row-Level Split
# ============================================

from sklearn.model_selection import train_test_split

X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("BEFORE — Random Row-Level Split")
print("--------------------------------")
print("Training rows:", len(X_train_before))
print("Test rows:", len(X_test_before))

print(
    "Training declining rate:",
    y_train_before.mean()
)

print(
    "Test declining rate:",
    y_test_before.mean()
)

BEFORE — Random Row-Level Split
--------------------------------
Training rows: 24000
Test rows: 6000
Training declining rate: 0.5420833333333334
Test declining rate: 0.542


In [19]:
# ============================================
# WEEK 6 — BEFORE: Model with Missing-Value Handling
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------
# 1. Numerical preprocessing
# --------------------------------------------

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# --------------------------------------------
# 2. Categorical preprocessing
# --------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# --------------------------------------------
# 3. Combine preprocessing
# --------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

# --------------------------------------------
# 4. Build model
# --------------------------------------------

model_before = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        )
    ]
)

# --------------------------------------------
# 5. Train
# --------------------------------------------

model_before.fit(
    X_train_before,
    y_train_before
)

# --------------------------------------------
# 6. Predictions
# --------------------------------------------

y_prob_before = model_before.predict_proba(
    X_test_before
)[:, 1]

y_pred_before = (
    y_prob_before >= 0.5
).astype(int)

# --------------------------------------------
# 7. Metrics
# --------------------------------------------

roc_auc_before = roc_auc_score(
    y_test_before,
    y_prob_before
)

average_precision_before = average_precision_score(
    y_test_before,
    y_prob_before
)

precision_before = precision_score(
    y_test_before,
    y_pred_before
)

recall_before = recall_score(
    y_test_before,
    y_pred_before
)

f1_before = f1_score(
    y_test_before,
    y_pred_before
)

# --------------------------------------------
# 8. Precision@50
# --------------------------------------------

top_50_indices_before = (
    y_prob_before.argsort()[-50:][::-1]
)

precision_at_50_before = (
    y_test_before.iloc[top_50_indices_before].sum() / 50
)

# --------------------------------------------
# 9. Results
# --------------------------------------------

print("BEFORE — Random Row-Level Split")
print("--------------------------------")
print(f"ROC AUC:           {roc_auc_before:.3f}")
print(f"Average Precision: {average_precision_before:.3f}")
print(f"Precision:         {precision_before:.3f}")
print(f"Recall:            {recall_before:.3f}")
print(f"F1 Score:          {f1_before:.3f}")
print(f"Precision@50:      {precision_at_50_before:.3f}")

BEFORE — Random Row-Level Split
--------------------------------
ROC AUC:           0.921
Average Precision: 0.938
Precision:         0.824
Recall:            0.867
F1 Score:          0.845
Precision@50:      1.000


In [20]:
# ============================================
# WEEK 6 — AFTER: Client-Grouped Split
# ============================================

from sklearn.model_selection import GroupShuffleSplit

# --------------------------------------------
# 1. Create client-grouped train/test split
# --------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=client_ids
    )
)

X_train_after = X.iloc[train_idx]
X_test_after = X.iloc[test_idx]

y_train_after = y.iloc[train_idx]
y_test_after = y.iloc[test_idx]

clients_train = client_ids.iloc[train_idx]
clients_test = client_ids.iloc[test_idx]


# --------------------------------------------
# 2. Verify client separation
# --------------------------------------------

shared_clients = set(
    clients_train
).intersection(
    set(clients_test)
)

print("AFTER — Client-Grouped Split")
print("----------------------------")
print("Training rows:", len(X_train_after))
print("Test rows:", len(X_test_after))

print("\nTraining clients:", clients_train.nunique())
print("Test clients:", clients_test.nunique())
print("Shared clients:", len(shared_clients))

print("\nTraining declining rate:",
      round(y_train_after.mean(), 4))

print("Test declining rate:",
      round(y_test_after.mean(), 4))


# --------------------------------------------
# 3. Create a FRESH model
# --------------------------------------------

model_after = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        )
    ]
)


# --------------------------------------------
# 4. Train on training clients only
# --------------------------------------------

model_after.fit(
    X_train_after,
    y_train_after
)


# --------------------------------------------
# 5. Generate predictions
# --------------------------------------------

y_prob_after = model_after.predict_proba(
    X_test_after
)[:, 1]

y_pred_after = (
    y_prob_after >= 0.5
).astype(int)


# --------------------------------------------
# 6. Calculate metrics
# --------------------------------------------

roc_auc_after = roc_auc_score(
    y_test_after,
    y_prob_after
)

average_precision_after = average_precision_score(
    y_test_after,
    y_prob_after
)

precision_after = precision_score(
    y_test_after,
    y_pred_after
)

recall_after = recall_score(
    y_test_after,
    y_pred_after
)

f1_after = f1_score(
    y_test_after,
    y_pred_after
)


# --------------------------------------------
# 7. Precision@50
# --------------------------------------------

top_50_indices_after = (
    y_prob_after.argsort()[-50:][::-1]
)

precision_at_50_after = (
    y_test_after.iloc[top_50_indices_after].sum() / 50
)


# --------------------------------------------
# 8. Display results
# --------------------------------------------

print("\nAFTER — Client-Grouped Performance")
print("----------------------------------")
print(f"ROC AUC:           {roc_auc_after:.3f}")
print(f"Average Precision: {average_precision_after:.3f}")
print(f"Precision:         {precision_after:.3f}")
print(f"Recall:            {recall_after:.3f}")
print(f"F1 Score:          {f1_after:.3f}")
print(f"Precision@50:      {precision_at_50_after:.3f}")

AFTER — Client-Grouped Split
----------------------------
Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7
Shared clients: 0

Training declining rate: 0.5501
Test declining rate: 0.511

AFTER — Client-Grouped Performance
----------------------------------
ROC AUC:           0.838
Average Precision: 0.851
Precision:         0.718
Recall:            0.810
F1 Score:          0.761
Precision@50:      1.000


The original random row-level split produced stronger measured performance than the client-grouped split. ROC AUC decreased from 0.921 to 0.838, while Average Precision decreased from 0.938 to 0.851. Precision, recall, and F1 also decreased under the client-grouped evaluation. The grouped split contained 25 training clients and 7 test clients, with zero clients shared between the two sets. This suggests that the original row-level evaluation provided a more favorable estimate of generalization across clients. The model nevertheless retained useful measured discrimination on the unseen-client test set, with ROC AUC of 0.838 and Average Precision of 0.851. Precision@50 was 1.000 in both evaluation

3. Leakage audit
The same hunt from Week 3, on your final feature set.

In [21]:
# ============================================
# WEEK 6 — SECTION 3
# LEAKAGE AUDIT
# ============================================

# --------------------------------------------
# 1. Features currently used by the model
# --------------------------------------------

print("FEATURES USED BY MODEL")
print("======================")

for i, feature in enumerate(X.columns, start=1):
    print(f"{i:2}. {feature}")


# --------------------------------------------
# 2. Direct label-derived features
# --------------------------------------------

label_derived_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nDIRECT LABEL-DERIVED FEATURES")
print("=============================")

for feature in label_derived_features:
    print(
        f"{feature}:",
        "PRESENT IN X" if feature in X.columns else "NOT IN X"
    )


# --------------------------------------------
# 3. Identifier features
# --------------------------------------------

identifier_features = [
    "content_id",
    "client_id"
]

print("\nIDENTIFIER FEATURES")
print("===================")

for feature in identifier_features:
    print(
        f"{feature}:",
        "PRESENT IN X" if feature in X.columns else "NOT IN X"
    )


# --------------------------------------------
# 4. Potentially overlapping time-window
#    features
# --------------------------------------------

time_window_features = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("\nTIME-WINDOW FEATURES")
print("====================")

for feature in time_window_features:
    print(
        f"{feature}:",
        "PRESENT IN X" if feature in X.columns else "NOT IN X"
    )


# --------------------------------------------
# 5. Check relationship with target
# --------------------------------------------

print("\nTARGET RELATIONSHIPS")
print("====================")

for feature in time_window_features:

    if feature in df.columns:

        grouped = df.groupby(
            "is_declining_label"
        )[feature].mean()

        print(f"\n{feature}")
        print(grouped)


# --------------------------------------------
# 6. Check missingness
# --------------------------------------------

print("\nMISSING VALUES IN MODEL FEATURES")
print("================================")

missing_counts = X.isna().sum()

missing_features = (
    missing_counts[
        missing_counts > 0
    ]
    .sort_values(ascending=False)
)

print(missing_features)


# --------------------------------------------
# 7. Summary
# --------------------------------------------

print("\nLEAKAGE AUDIT SUMMARY")
print("=====================")

print(
    "Direct label-derived features removed:",
    all(
        feature not in X.columns
        for feature in label_derived_features
    )
)

print(
    "Identifiers removed from model:",
    all(
        feature not in X.columns
        for feature in identifier_features
    )
)

print(
    "Time-window features still present:",
    [
        feature
        for feature in time_window_features
        if feature in X.columns
    ]
)

FEATURES USED BY MODEL
 1. search_volume
 2. competition
 3. competition_level
 4. cpc
 5. content_type
 6. main_intent
 7. word_count
 8. char_count
 9. provider_used
10. model_used
11. impressions_90d
12. clicks_90d
13. pageviews_90d
14. sessions_90d
15. users_90d
16. engaged_sessions_90d
17. ai_sessions_90d
18. scroll_events_90d
19. days_with_impressions
20. days_with_sessions
21. impressions_last_30d
22. clicks_last_30d
23. sessions_last_30d
24. impressions_prev_30d
25. clicks_prev_30d
26. sessions_prev_30d
27. content_age_days
28. age_tier
29. age_tier_order
30. days_since_last_update
31. freshness_tier
32. word_count_tier
33. char_count_tier
34. ctr
35. avg_position
36. engagement_rate
37. scroll_rate
38. ai_traffic_pct
39. impression_tier
40. position_tier

DIRECT LABEL-DERIVED FEATURES
trend_direction: NOT IN X
trend_pct: NOT IN X
is_declining_label: NOT IN X

IDENTIFIER FEATURES
content_id: NOT IN X
client_id: NOT IN X

TIME-WINDOW FEATURES
impressions_last_30d: PRESENT IN X
c

In [22]:
# ============================================
# WEEK 6 — LEAKAGE TIMING CHECK
# ============================================

# Show the relevant columns together
timing_columns = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("Relevant timing / trend columns:")
print("---------------------------------")

display(
    df[timing_columns].head(10)
)


# --------------------------------------------
# Compare the last and previous 30-day metrics
# --------------------------------------------

print("\nExample differences between periods:")
print("--------------------------------------")

df["impression_change"] = (
    df["impressions_last_30d"]
    - df["impressions_prev_30d"]
)

df["click_change"] = (
    df["clicks_last_30d"]
    - df["clicks_prev_30d"]
)

df["session_change"] = (
    df["sessions_last_30d"]
    - df["sessions_prev_30d"]
)

display(
    df[
        [
            "trend_direction",
            "trend_pct",
            "impressions_last_30d",
            "impressions_prev_30d",
            "impression_change",
            "clicks_last_30d",
            "clicks_prev_30d",
            "click_change",
            "sessions_last_30d",
            "sessions_prev_30d",
            "session_change"
        ]
    ].head(10)
)

Relevant timing / trend columns:
---------------------------------


,trend_direction,trend_pct,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d
0,down,-41.4,578,2,2,987,13,9
1,down,-57.7,2501,2,3,5915,1,2
2,down,-60.9,2382,1,1,6089,3,3
3,stable,-13.8,3626,22,35,4206,17,26
4,down,-34.7,4211,10,14,6452,2,9
5,down,-38.9,617,0,4,1009,1,1
6,down,-92.3,1,0,0,13,0,1
7,stable,0.6,636,1,24,632,0,4
8,down,-58.8,5696,9,36,13828,8,14
9,down,-29.2,252,0,0,356,0,0



Example differences between periods:
--------------------------------------


,trend_direction,trend_pct,impressions_last_30d,impressions_prev_30d,impression_change,clicks_last_30d,clicks_prev_30d,click_change,sessions_last_30d,sessions_prev_30d,session_change
0,down,-41.4,578,987,-409,2,13,-11,2,9,-7
1,down,-57.7,2501,5915,-3414,2,1,1,3,2,1
2,down,-60.9,2382,6089,-3707,1,3,-2,1,3,-2
3,stable,-13.8,3626,4206,-580,22,17,5,35,26,9
4,down,-34.7,4211,6452,-2241,10,2,8,14,9,5
5,down,-38.9,617,1009,-392,0,1,-1,4,1,3
6,down,-92.3,1,13,-12,0,0,0,0,1,-1
7,stable,0.6,636,632,4,1,0,1,24,4,20
8,down,-58.8,5696,13828,-8132,9,8,1,36,14,22
9,down,-29.2,252,356,-104,0,0,0,0,0,0


| Feature group          | Status                              | Reason                                                                 |
| ---------------------- | ----------------------------------- | ---------------------------------------------------------------------- |
| `trend_direction`      | Removed                             | Directly used to construct `is_declining_label`                        |
| `trend_pct`            | Removed                             | Used to derive `trend_direction`, which defines the label              |
| `content_id`           | Removed                             | Identifier; not a meaningful predictive feature                        |
| `client_id`            | Removed from X                      | Identifier; retained separately only for grouped validation            |
| `impressions_last_30d` | **Leakage risk**                    | Represents the outcome-period performance used in the trend comparison |
| `clicks_last_30d`      | **Leakage risk**                    | Represents outcome-period performance used in the trend comparison     |
| `sessions_last_30d`    | **Leakage risk**                    | Represents outcome-period performance used in the trend comparison     |
| `impressions_prev_30d` | **Leakage risk**                    | Represents one of the periods used in the trend comparison             |
| `clicks_prev_30d`      | **Leakage risk**                    | Represents one of the periods used in the trend comparison             |
| `sessions_prev_30d`    | **Leakage risk**                    | Represents one of the periods used in the trend comparison             |
| Other features         | No direct leakage identified so far | Require normal feature-timing and semantic review                      |


The audit identified six performance-window features as leakage-risk for the current prediction setup because they represent the same last-30-day and previous-30-day periods used to derive the declining outcome. Although these features are predictive of the label, their timing means they would not provide a clean test of predicting decline before the outcome period. Therefore, their inclusion can make model performance appear stronger than what would be achievable using strictly pre-outcome information.

In [23]:
# ============================================
# WEEK 6 — SECTION 3
# CLEAN MODEL WITHOUT LEAKAGE-RISK FEATURES
# ============================================

# --------------------------------------------
# 1. Define leakage-risk features
# --------------------------------------------

leakage_risk_features = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("Removing the following leakage-risk features:")
for feature in leakage_risk_features:
    print("-", feature)


# --------------------------------------------
# 2. Create cleaned feature set
# --------------------------------------------

X_clean = X.drop(
    columns=leakage_risk_features
)

print("\nOriginal number of features:", X.shape[1])
print("Cleaned number of features:", X_clean.shape[1])


# --------------------------------------------
# 3. Re-identify feature types
# --------------------------------------------

categorical_features_clean = (
    X_clean
    .select_dtypes(include=["object"])
    .columns
    .tolist()
)

numerical_features_clean = (
    X_clean
    .select_dtypes(exclude=["object"])
    .columns
    .tolist()
)

print("\nCategorical features:",
      len(categorical_features_clean))

print("Numerical features:",
      len(numerical_features_clean))


# --------------------------------------------
# 4. Build clean preprocessing pipeline
# --------------------------------------------

numerical_pipeline_clean = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline_clean = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor_clean = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline_clean,
            numerical_features_clean
        ),
        (
            "cat",
            categorical_pipeline_clean,
            categorical_features_clean
        )
    ]
)


# --------------------------------------------
# 5. Create clean Logistic Regression model
# --------------------------------------------

model_clean = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_clean
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        )
    ]
)


# --------------------------------------------
# 6. Use the SAME random split indices
# --------------------------------------------
# This is important:
# We are changing the FEATURES, not the test data.

X_train_clean = X_clean.iloc[
    X_train_before.index
]

X_test_clean = X_clean.iloc[
    X_test_before.index
]


# --------------------------------------------
# 7. Train clean model
# --------------------------------------------

model_clean.fit(
    X_train_clean,
    y_train_before
)


# --------------------------------------------
# 8. Generate predictions
# --------------------------------------------

y_prob_clean = model_clean.predict_proba(
    X_test_clean
)[:, 1]

y_pred_clean = (
    y_prob_clean >= 0.5
).astype(int)


# --------------------------------------------
# 9. Calculate metrics
# --------------------------------------------

roc_auc_clean = roc_auc_score(
    y_test_before,
    y_prob_clean
)

average_precision_clean = (
    average_precision_score(
        y_test_before,
        y_prob_clean
    )
)

precision_clean = precision_score(
    y_test_before,
    y_pred_clean
)

recall_clean = recall_score(
    y_test_before,
    y_pred_clean
)

f1_clean = f1_score(
    y_test_before,
    y_pred_clean
)


# --------------------------------------------
# 10. Precision@50
# --------------------------------------------

top_50_indices_clean = (
    y_prob_clean.argsort()[-50:][::-1]
)

precision_at_50_clean = (
    y_test_before
    .iloc[top_50_indices_clean]
    .sum() / 50
)


# --------------------------------------------
# 11. Display results
# --------------------------------------------

print("\nCLEAN MODEL — Random Row-Level Split")
print("------------------------------------")

print(f"ROC AUC:           {roc_auc_clean:.3f}")
print(f"Average Precision: {average_precision_clean:.3f}")
print(f"Precision:         {precision_clean:.3f}")
print(f"Recall:            {recall_clean:.3f}")
print(f"F1 Score:          {f1_clean:.3f}")
print(f"Precision@50:      {precision_at_50_clean:.3f}")

Removing the following leakage-risk features:
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d

Original number of features: 40
Cleaned number of features: 34

Categorical features: 11
Numerical features: 23

CLEAN MODEL — Random Row-Level Split
------------------------------------
ROC AUC:           0.700
Average Precision: 0.705
Precision:         0.659
Recall:            0.737
F1 Score:          0.696
Precision@50:      0.800


Empirical Leakage-Risk Check

To assess the practical impact of the identified leakage-risk features, I retrained the model after removing the six last-30-day and previous-30-day performance features. Using the same random train/test split, ROC AUC decreased from 0.921 to 0.700, Average Precision from 0.938 to 0.705, and Precision@50 from 1.000 to 0.800. This substantial reduction indicates that the removed features contributed considerable predictive information. Because these features represent the same periods used to derive the declining outcome, their predictive contribution should not be interpreted as clean evidence of pre-outcome prediction performance.


In [24]:
# ============================================
# WEEK 6 — SECTION 3
# FAILURE ANALYSIS
# ============================================

# --------------------------------------------
# 1. Create a table of test predictions
# --------------------------------------------

failure_df = X_test_clean.copy()

failure_df["actual_label"] = y_test_before.values
failure_df["predicted_probability"] = y_prob_clean
failure_df["predicted_label"] = y_pred_clean

# --------------------------------------------
# 2. Identify incorrect predictions
# --------------------------------------------

failure_df["correct"] = (
    failure_df["actual_label"]
    == failure_df["predicted_label"]
)

errors = failure_df[
    ~failure_df["correct"]
].copy()

print("Total test examples:", len(failure_df))
print("Incorrect predictions:", len(errors))
print(
    "Error rate:",
    round(len(errors) / len(failure_df), 3)
)


# --------------------------------------------
# 3. False positives
# --------------------------------------------

false_positives = errors[
    (errors["actual_label"] == 0)
    &
    (errors["predicted_label"] == 1)
].copy()

# --------------------------------------------
# 4. False negatives
# --------------------------------------------

false_negatives = errors[
    (errors["actual_label"] == 1)
    &
    (errors["predicted_label"] == 0)
].copy()


print("\nFalse positives:", len(false_positives))
print("False negatives:", len(false_negatives))


# --------------------------------------------
# 5. Show highest-confidence false positives
# --------------------------------------------

print("\nTOP FALSE POSITIVES")
print("===================")

fp_display = false_positives.sort_values(
    "predicted_probability",
    ascending=False
)

display(
    fp_display[
        [
            "predicted_probability",
            "actual_label",
            "predicted_label",
            "content_type",
            "main_intent",
            "content_age_days",
            "days_since_last_update",
            "word_count",
            "avg_position",
            "ctr"
        ]
    ].head(10)
)


# --------------------------------------------
# 6. Show highest-confidence false negatives
# --------------------------------------------

print("\nTOP FALSE NEGATIVES")
print("===================")

fn_display = false_negatives.sort_values(
    "predicted_probability",
    ascending=True
)

display(
    fn_display[
        [
            "predicted_probability",
            "actual_label",
            "predicted_label",
            "content_type",
            "main_intent",
            "content_age_days",
            "days_since_last_update",
            "word_count",
            "avg_position",
            "ctr"
        ]
    ].head(10)
)

Total test examples: 6000
Incorrect predictions: 2097
Error rate: 0.349

False positives: 1243
False negatives: 854

TOP FALSE POSITIVES


,predicted_probability,actual_label,predicted_label,content_type,main_intent,content_age_days,days_since_last_update,word_count,avg_position,ctr
1865,0.859627,0,1,keyword article,informational,139,104,8423.0,12.5,0.16
26544,0.857297,0,1,keyword article,informational,144,20,5654.0,15.4,0.03
3868,0.854745,0,1,keyword article,informational,90,20,2628.0,5.5,0.16
22840,0.854291,0,1,keyword article,informational,165,104,1602.0,16.8,0.00
16643,0.854121,0,1,keyword article,informational,165,104,1672.0,26.5,0.00
15342,0.854084,0,1,keyword article,informational,144,20,5974.0,11.7,0.13
2308,0.846672,0,1,keyword article,informational,174,104,1397.0,21.3,0.00
1247,0.844755,0,1,keyword article,commercial,264,104,1612.0,15.7,0.37
18531,0.841403,0,1,keyword article,informational,348,104,1914.0,16.1,0.60
17238,0.840077,0,1,keyword article,informational,229,104,6814.0,19.5,0.38



TOP FALSE NEGATIVES


,predicted_probability,actual_label,predicted_label,content_type,main_intent,content_age_days,days_since_last_update,word_count,avg_position,ctr
12977,0.035686,1,0,keyword article,transactional,517,11,2937.0,80.2,0.00
27614,0.076589,1,0,feedly article,NaN,281,20,858.0,2.5,25.00
7412,0.078335,1,0,feedly article,NaN,293,20,689.0,1.7,0.00
22347,0.081417,1,0,keyword article,informational,495,20,NaN,90.0,0.00
25107,0.088410,1,0,keyword article,commercial,144,104,5402.0,20.3,0.86
17896,0.093333,1,0,keyword article,informational,517,22,NaN,63.8,0.00
5608,0.093880,1,0,feedly article,NaN,290,20,837.0,2.7,0.00
12864,0.110488,1,0,feedly article,NaN,294,20,978.0,2.0,0.00
21819,0.116584,1,0,keyword article,informational,445,20,3097.0,2.3,0.41
3700,0.120383,1,0,feedly article,NaN,309,20,800.0,2.5,0.00


Failure Analysis

The cleaned model produced 2,097 incorrect predictions out of 6,000 test
, corresponding to an error rate of 34.9%. There were 1,243 false positives and 854 false negatives. Inspection of high-confidence errors showed that both error types occur across different content characteristics. Several false negatives were older pages, including examples aged 445–517 days, with some having very low average positions and zero CTR. False positives included several keyword articles, often informational, with substantial word counts and varying content ages. These examples show that the available pre-outcome features do not perfectly distinguish declining from non-declining pages and that individual predictions can be uncertain even when the model assigns high or low probabilities.

In [25]:
# ============================================
# WEEK 6 — SECTION 3
# CLEAN MODEL + CLIENT-GROUPED SPLIT
# ============================================

# --------------------------------------------
# 1. Create clean grouped train/test data
# --------------------------------------------

X_train_clean_grouped = X_clean.iloc[train_idx]
X_test_clean_grouped = X_clean.iloc[test_idx]

y_train_clean_grouped = y.iloc[train_idx]
y_test_clean_grouped = y.iloc[test_idx]


# --------------------------------------------
# 2. Build preprocessing pipeline
# --------------------------------------------

numerical_pipeline_final = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline_final = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor_final = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline_final,
            numerical_features_clean
        ),
        (
            "cat",
            categorical_pipeline_final,
            categorical_features_clean
        )
    ]
)


# --------------------------------------------
# 3. Build final clean model
# --------------------------------------------

model_final = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_final
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        )
    ]
)


# --------------------------------------------
# 4. Train on unseen-client setup
# --------------------------------------------

model_final.fit(
    X_train_clean_grouped,
    y_train_clean_grouped
)


# --------------------------------------------
# 5. Predictions
# --------------------------------------------

y_prob_final = model_final.predict_proba(
    X_test_clean_grouped
)[:, 1]

y_pred_final = (
    y_prob_final >= 0.5
).astype(int)


# --------------------------------------------
# 6. Calculate metrics
# --------------------------------------------

roc_auc_final = roc_auc_score(
    y_test_clean_grouped,
    y_prob_final
)

average_precision_final = (
    average_precision_score(
        y_test_clean_grouped,
        y_prob_final
    )
)

precision_final = precision_score(
    y_test_clean_grouped,
    y_pred_final
)

recall_final = recall_score(
    y_test_clean_grouped,
    y_pred_final
)

f1_final = f1_score(
    y_test_clean_grouped,
    y_pred_final
)


# --------------------------------------------
# 7. Precision@50
# --------------------------------------------

top_50_indices_final = (
    y_prob_final.argsort()[-50:][::-1]
)

precision_at_50_final = (
    y_test_clean_grouped
    .iloc[top_50_indices_final]
    .sum() / 50
)


# --------------------------------------------
# 8. Display final audited results
# --------------------------------------------

print("FINAL AUDITED MODEL")
print("===================")

print("Evaluation: Client-grouped")
print("Leakage-risk features: Removed")
print("Shared clients: 0")

print("\nPerformance:")
print(f"ROC AUC:           {roc_auc_final:.3f}")
print(f"Average Precision: {average_precision_final:.3f}")
print(f"Precision:         {precision_final:.3f}")
print(f"Recall:            {recall_final:.3f}")
print(f"F1 Score:          {f1_final:.3f}")
print(f"Precision@50:      {precision_at_50_final:.3f}")

FINAL AUDITED MODEL
Evaluation: Client-grouped
Leakage-risk features: Removed
Shared clients: 0

Performance:
ROC AUC:           0.577
Average Precision: 0.569
Precision:         0.561
Recall:            0.652
F1 Score:          0.603
Precision@50:      0.640


After removing the six identified leakage-risk performance-window features, I evaluated the model using a client-grouped split so that no client appeared in both training and testing. The resulting test set contained 7 unseen clients, with 0 clients shared between training and testing. Measured ROC AUC decreased to 0.577, Average Precision to 0.569, and Precision@50 to 0.640. Compared with the original random-split model, this indicates that the original performance estimate was substantially more favorable than the performance measured under the stricter feature and validation setup.

 Claim rewrite
Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.

Claim 1 — Learned model vs. baseline

Original claim

The learned model does not beat the Week-4 baseline.

Evidence

In the Week-5 evaluation, the Random Forest achieved Precision@50 of 0.660, compared with 0.840 for the Week-4 baseline.

Revised claim

In the Week-5 evaluation, the Random Forest achieved a Precision@50 of 0.660, below the Week-4 baseline of 0.840. Therefore, the learned Random Forest model did not outperform the baseline on the measured Precision@50 metric in this evaluation.

Claim 2 — Random Forest false positives

Original claim

The Random Forest produced 17 false positives among its top 50 ranked pages. Several high-scoring false positives were actually stable or improving pages. This suggests that the model can interpret strong visibility and performance-related signals as decline risk even when the actual trend is not declining.

Revised claim

Among the Random Forest's top 50 ranked pages in the Week-5 evaluation, 17 were false positives. Several high-scoring false positives were labeled as stable or improving. This indicates that the model sometimes ranked pages with strong visibility or performance-related characteristics as higher decline-risk even when their observed trend label was not declining.

Self-check
Before you submit, confirm each line honestly:

 Every section above is filled — markdown thinking AND the code that backs it
 The notebook runs top to bottom with no errors (Runtime → Run all)
 No client names, URLs, or private queries anywhere
 My claims use careful words: observed, measured, directional, decision-support
 Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.
